In [1]:
import datasets

/share/home/zxwang/miniconda3/envs/deepagent/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### GSM8K

In [2]:
data_path = "/share/home/sxjiang/myproject/LHTIR/data/gsm8k/train.parquet"
dataset_gsm8k = datasets.load_dataset("parquet", data_files=data_path)
data_item_gsm8k = dataset_gsm8k['train'][0]

In [3]:
data_item_gsm8k

{'data_source': 'openai/gsm8k',
 'agent_name': 'tool_agent',
 'prompt': [{'content': 'You are a math expert. You are given a question and you need to solve it step by step. Reasoning step by step before any tool call. You should use the `calc_gsm8k_reward` tool after step by step solving the question, before generate final answer at least once and refine your answer if necessary. Put your final answer in the format of `#### <answer>`.',
   'role': 'system'},
  {'content': "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May? Let's think step by step and output the final answer after `####`.",
   'role': 'user'}],
 'ability': 'math',
 'reward_model': {'ground_truth': '72', 'style': 'rule'},
 'extra_info': {'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72',
  'index': 0,
  'interaction_kwargs': {'

#### FTRL

In [4]:
data_path = "/share/home/sxjiang/myproject/LHTIR/data/ftrl/with_agent_name/train.parquet"
dataset_ftrl = datasets.load_dataset("parquet", data_files=data_path)
data_item_ftrl = dataset_ftrl['train'][0]


In [5]:
data_item_ftrl.keys()

dict_keys(['messages', 'tools', 'codes', 'unsolved_set', 'solve_rate', 'data_source', 'split', 'answer', 'ground_truth', 'agent_name'])

In [6]:
import json
from copy import deepcopy


def format_string_dicts_recursive(obj):
    """
    递归地将 data_item 中所有「字符串形式的 dict/list」解析为真正的 dict/list。
    会就地修改传入对象，若需保留原对象请先 deepcopy。
    """
    if isinstance(obj, dict):
        for key in list(obj.keys()):
            obj[key] = format_string_dicts_recursive(obj[key])
        return obj
    if isinstance(obj, list):
        for i in range(len(obj)):
            obj[i] = format_string_dicts_recursive(obj[i])
        return obj
    if isinstance(obj, str):
        s = obj.strip()
        # 只尝试看起来像 JSON 对象或数组的字符串（以 { 或 [ 开头）
        if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
            try:
                parsed = json.loads(obj)
                if isinstance(parsed, (dict, list)):
                    return format_string_dicts_recursive(parsed)
            except (json.JSONDecodeError, TypeError):
                pass
    return obj


# 使用示例（不修改原 data_item，返回新对象）：
# data_item_formatted = format_string_dicts_recursive(deepcopy(data_item_ftrl))
# 之后可直接用 data_item_formatted['tools']、data_item_formatted['codes'] 等，已是 dict/list

In [7]:
import json
print(json.loads(data_item_ftrl['codes'])["book_publication_info_retriever"])

def book_publication_info_retriever(title, author=None, edition=None, language=None, include_translations=False, isbn=None, publication_year_range=None, genre=None, publisher=None):
    """
    Retrieve comprehensive bibliographic information about books.

    Parameters:
    - title (str): The title of the book.
    - author (str, optional): The author of the book.
    - edition (str, optional): Specific edition of the book.
    - language (str, optional): The language of the book.
    - include_translations (bool, optional): Include translated editions. Default is False.
    - isbn (str, optional): The International Standard Book Number.
    - publication_year_range (dict, optional): The range of publication years with 'start_year' and 'end_year'.
    - genre (str, optional): The genre of the book.
    - publisher (str, optional): The publisher of the book.

    Returns:
    - dict: A dictionary containing bibliographic information or an error message.
    """
    # Error handling fo

In [8]:
json.loads(data_item_ftrl['tools'])

[{'type': 'function',
  'function': {'name': 'book_publication_info_retriever',
   'description': 'A tool designed to retrieve comprehensive bibliographic information about books, including publication dates, authors, publishers, editions, and more, from a detailed database.',
   'parameters': {'type': 'object',
    'properties': {'title': {'type': 'string',
      'description': 'The title of the book for which publication information is being requested.'},
     'author': {'type': 'string',
      'description': 'The author of the book to help narrow down search results, especially for books with common titles. Optional.'},
     'edition': {'type': 'string',
      'description': 'Specific edition of the book if applicable. Optional.'},
     'language': {'type': 'string',
      'description': 'The language of the book to filter results, especially for books available in multiple languages. Optional.'},
     'include_translations': {'type': 'boolean',
      'description': 'Whether to incl

In [9]:
len(dataset_ftrl['train'])

2215

In [10]:
for i in range(len(dataset_ftrl['train'])):
    data_item_ftrl = dataset_ftrl['train'][i]
    if "codes" not in data_item_ftrl.keys():
        print(data_item_ftrl.keys())
        break
data_item_ftrl=dataset_ftrl['train'][0]

In [11]:
print(json.dumps(format_string_dicts_recursive(data_item_ftrl),indent=4))

{
    "messages": [
        {
            "content": "Please call given tools to answer the question. Please note that all your information must be obtained by calling tools and not by answering the question directly.\nIf the call fails, you need to try to correct it and continue until you arrive at an answer. Only output the final answer (in words, numbers or phrase) inside the <answer></answer> tag, without any explanations or extra information.\nQuestion: Which party remained the largest in the Belgian federal election on June 9, 2024?",
            "role": "user"
        }
    ],
    "tools": [
        {
            "type": "function",
            "function": {
                "name": "book_publication_info_retriever",
                "description": "A tool designed to retrieve comprehensive bibliographic information about books, including publication dates, authors, publishers, editions, and more, from a detailed database.",
                "parameters": {
                    "typ

In [12]:
data_path = "/share/home/sxjiang/myproject/LHTIR/data/ftrl/with_agent_name_mask/train.parquet"
dataset_ftrl_mask = datasets.load_dataset("parquet", data_files=data_path)
data_item_ftrl_mask = dataset_ftrl_mask['train'][0]


Generating train split: 2215 examples [00:00, 9527.98 examples/s]


In [13]:
print(json.dumps(format_string_dicts_recursive(data_item_ftrl)["codes"],indent=4))

{
    "book_publication_info_retriever": "def book_publication_info_retriever(title, author=None, edition=None, language=None, include_translations=False, isbn=None, publication_year_range=None, genre=None, publisher=None):\n    \"\"\"\n    Retrieve comprehensive bibliographic information about books.\n\n    Parameters:\n    - title (str): The title of the book.\n    - author (str, optional): The author of the book.\n    - edition (str, optional): Specific edition of the book.\n    - language (str, optional): The language of the book.\n    - include_translations (bool, optional): Include translated editions. Default is False.\n    - isbn (str, optional): The International Standard Book Number.\n    - publication_year_range (dict, optional): The range of publication years with 'start_year' and 'end_year'.\n    - genre (str, optional): The genre of the book.\n    - publisher (str, optional): The publisher of the book.\n\n    Returns:\n    - dict: A dictionary containing bibliographic inf

In [14]:
data_item_ftrl_mask.keys()

dict_keys(['messages', 'tools', 'codes', 'unsolved_set', 'solve_rate', 'data_source', 'split', 'answer', 'ground_truth', 'agent_name'])

In [15]:
print(json.dumps(format_string_dicts_recursive(data_item_ftrl_mask)["codes"],indent=4))

{
    "ue1jfd": "def ue1jfd(title, author=None, edition=None, language=None, include_translations=False, isbn=None, publication_year_range=None, genre=None, publisher=None):\n    \"\"\"\n    Retrieve comprehensive bibliographic information about books.\n\n    Parameters:\n    - title (str): The title of the book.\n    - author (str, optional): The author of the book.\n    - edition (str, optional): Specific edition of the book.\n    - language (str, optional): The language of the book.\n    - include_translations (bool, optional): Include translated editions. Default is False.\n    - isbn (str, optional): The International Standard Book Number.\n    - publication_year_range (dict, optional): The range of publication years with 'start_year' and 'end_year'.\n    - genre (str, optional): The genre of the book.\n    - publisher (str, optional): The publisher of the book.\n\n    Returns:\n    - dict: A dictionary containing bibliographic information or an error message.\n    \"\"\"\n    # E

In [37]:
def call_function(name, arguments, code, **kwargs):
    namespace = {}
    exec(code, namespace, namespace)
    if name in namespace:
        predict = namespace[name](**arguments, **kwargs)
    else:
        raise NameError(f"name {name} is not defined")
    if type(predict) == dict or type(predict) == list:
        predict = json.dumps(predict, ensure_ascii=False)
    elif type(predict) != str:
        predict = str(predict)
    return predict
code = format_string_dicts_recursive(data_item_ftrl_mask)["codes"]["ftjrj8"]
arguments = {'election_date': '2024-06-09',
   'election_type': 'federal',
   'country': 'Belgium'}
call_function("ftjrj8", arguments, code)


'The New Flemish Alliance remained the largest party in the Belgian federal election on June 9, 2024.'

In [35]:
data_item_ftrl_mask["ground_truth"]

[{'name': 'ftjrj8',
  'parameters': {'election_date': '2024-06-09',
   'election_type': 'federal',
   'country': 'Belgium'}}]

#### ReCall

In [17]:
data_path = "/share/home/sxjiang/myproject/LHTIR/data/recall/train.parquet"
dataset_recall = datasets.load_dataset("parquet", data_files=data_path)
data_item_recall = dataset_recall['train'][0]

In [18]:
print(json.dumps(format_string_dicts_recursive(data_item_recall),indent=4))

{
    "data_source": "syntool_re_call",
    "question": "Plan a 3-night trip to Paris departing on 2023-10-15. Find the cheapest flight and a hotel under $150 per night within 2km of Eiffel Tower. What is the total cost?",
    "ability": "re_call",
    "reward_model": {
        "ground_truth": [
            "$720"
        ],
        "style": "rule"
    },
    "extra_info": {
        "env": "# Mocked Environment for Travel Planning\nfrom typing import List, Dict\n\n# Mocked Data\nflights = [\n    {\"flight_id\": 101, \"destination\": \"Paris\", \"date\": \"2023-10-15\", \"price\": 300},\n    {\"flight_id\": 102, \"destination\": \"Paris\", \"date\": \"2023-10-15\", \"price\": 280},\n    {\"flight_id\": 103, \"destination\": \"Paris\", \"date\": \"2023-10-16\", \"price\": 250}\n]\n\nhotels = [\n    {\"hotel_id\": 201, \"name\": \"River View\", \"destination\": \"Paris\", \"price_per_night\": 140, \"latitude\": 48.8588, \"longitude\": 2.2945},\n    {\"hotel_id\": 202, \"name\": \"Downtown

In [19]:
print(data_item_recall['extra_info']['env'])

# Mocked Environment for Travel Planning
from typing import List, Dict

# Mocked Data
flights = [
    {"flight_id": 101, "destination": "Paris", "date": "2023-10-15", "price": 300},
    {"flight_id": 102, "destination": "Paris", "date": "2023-10-15", "price": 280},
    {"flight_id": 103, "destination": "Paris", "date": "2023-10-16", "price": 250}
]

hotels = [
    {"hotel_id": 201, "name": "River View", "destination": "Paris", "price_per_night": 140, "latitude": 48.8588, "longitude": 2.2945},
    {"hotel_id": 202, "name": "Downtown Inn", "destination": "Paris", "price_per_night": 160, "latitude": 48.8566, "longitude": 2.3522},
    {"hotel_id": 203, "name": "Eiffel Stay", "destination": "Paris", "price_per_night": 130, "latitude": 48.8579, "longitude": 2.2965}
]

landmarks = [
    {"name": "Eiffel Tower", "latitude": 48.8584, "longitude": 2.2945}
]

# Mocked Functions
def get_available_flights(destination: str, date: str) -> List[Dict]:
    """Return flights matching destination and dat

In [20]:
print(data_item_recall['extra_info']['func_schemas'])

[{'type': 'function', 'function': {'name': 'get_available_flights', 'description': 'Return flights matching destination and date.', 'parameters': {'type': 'object', 'properties': {'destination': {'type': 'string', 'description': 'Destination city for the flight.'}, 'date': {'type': 'string', 'description': 'Date of the flight in YYYY-MM-DD format.'}}, 'required': ['destination', 'date']}}}, {'type': 'function', 'function': {'name': 'get_hotels_in_budget', 'description': 'Return hotels under max price per night.', 'parameters': {'type': 'object', 'properties': {'destination': {'type': 'string', 'description': 'Destination city for the hotel.'}, 'max_price_per_night': {'type': 'number', 'description': 'Maximum price per night for the hotel.'}}, 'required': ['destination', 'max_price_per_night']}}}, {'type': 'function', 'function': {'name': 'calculate_distance_from_landmark', 'description': 'Calculate Euclidean distance between hotel and landmark (simplified).', 'parameters': {'type': 'ob

#### AWM

In [21]:
data_path = "/share/home/sxjiang/dataset/AgentWorldModel-1K/gen_db.jsonl"
dataset_awm_db = datasets.load_dataset("json", data_files=data_path)
data_item_awm_db = dataset_awm_db['train'][0]
print(data_item_awm_db)

{'scenario': 'e_commerce_33', 'db_schema': {'tables': [{'name': 'users', 'ddl': 'CREATE TABLE users (\n    id INTEGER PRIMARY KEY,\n    username TEXT UNIQUE NOT NULL,\n    email TEXT UNIQUE NOT NULL,\n    full_name TEXT,\n    created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,\n    updated_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP\n);', 'indexes': ['CREATE INDEX idx_users_email ON users(email);']}, {'name': 'product_categories', 'ddl': 'CREATE TABLE product_categories (\n    id INTEGER PRIMARY KEY,\n    name TEXT NOT NULL,\n    parent_id INTEGER,\n    created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,\n    updated_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,\n    FOREIGN KEY (parent_id) REFERENCES product_categories(id)\n);', 'indexes': ['CREATE UNIQUE INDEX idx_product_categories_name ON product_categories(name);', 'CREATE INDEX idx_product_categories_parent_id ON product_categories(parent_id);']}, {'name': 'products', 'ddl': 'CREATE TABLE products (\n    id INTEGER P

In [22]:
print(json.dumps(data_item_awm_db,indent=4))

{
    "scenario": "e_commerce_33",
    "db_schema": {
        "tables": [
            {
                "name": "users",
                "ddl": "CREATE TABLE users (\n    id INTEGER PRIMARY KEY,\n    username TEXT UNIQUE NOT NULL,\n    email TEXT UNIQUE NOT NULL,\n    full_name TEXT,\n    created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,\n    updated_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP\n);",
                "indexes": [
                    "CREATE INDEX idx_users_email ON users(email);"
                ]
            },
            {
                "name": "product_categories",
                "ddl": "CREATE TABLE product_categories (\n    id INTEGER PRIMARY KEY,\n    name TEXT NOT NULL,\n    parent_id INTEGER,\n    created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,\n    updated_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,\n    FOREIGN KEY (parent_id) REFERENCES product_categories(id)\n);",
                "indexes": [
                    "CREATE UNIQUE INDEX i

In [23]:
data_path = "/share/home/sxjiang/dataset/AgentWorldModel-1K/gen_envs.jsonl"
dataset_awm = datasets.load_dataset("json", data_files=data_path)
data_item_awm_env = dataset_awm['train'][0]
print(json.dumps(data_item_awm_env,indent=4))

{
    "scenario": "content_platform_1",
    "db_path": "./outputs/databases/content_platform_1.db",
    "full_code": "from fastapi import FastAPI, Query, Path, Body\nfrom pydantic import BaseModel, Field, ConfigDict\nfrom typing import Optional, List, Dict\nfrom sqlalchemy import (\n    create_engine,\n    Column,\n    Integer,\n    String,\n    Text,\n    DateTime,\n    ForeignKey,\n    CheckConstraint,\n    UniqueConstraint,\n    Index,\n    Float,\n)\nfrom sqlalchemy.orm import declarative_base, relationship, sessionmaker\nfrom datetime import datetime\nimport os\n\ndatabase_url = os.getenv(\"DATABASE_PATH\", \"sqlite:///outputs/databases/youtube.db\")\n\nengine = create_engine(database_url, connect_args={\"check_same_thread\": False} if database_url.startswith(\"sqlite\") else {})\nSessionLocal = sessionmaker(bind=engine, autoflush=False, autocommit=False)\n\nBase = declarative_base()\n\n\nclass User(Base):\n    __tablename__ = \"users\"\n    id = Column(Integer, primary_key=True)\

In [24]:
print(data_item_awm_env["full_code"])

from fastapi import FastAPI, Query, Path, Body
from pydantic import BaseModel, Field, ConfigDict
from typing import Optional, List, Dict
from sqlalchemy import (
    create_engine,
    Column,
    Integer,
    String,
    Text,
    DateTime,
    ForeignKey,
    CheckConstraint,
    UniqueConstraint,
    Index,
    Float,
)
from sqlalchemy.orm import declarative_base, relationship, sessionmaker
from datetime import datetime
import os

database_url = os.getenv("DATABASE_PATH", "sqlite:///outputs/databases/youtube.db")

engine = create_engine(database_url, connect_args={"check_same_thread": False} if database_url.startswith("sqlite") else {})
SessionLocal = sessionmaker(bind=engine, autoflush=False, autocommit=False)

Base = declarative_base()


class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True)
    username = Column(String, unique=True, nullable=False, index=True)
    email = Column(String, unique=True, nullable=False, index=True)
    full_name = Col

In [25]:
data_path = "/share/home/sxjiang/dataset/AgentWorldModel-1K/gen_sample.jsonl"
dataset_awm_sample = datasets.load_dataset("json", data_files=data_path)
data_item_awm_sample = dataset_awm_sample['train'][0]
print(json.dumps(data_item_awm_sample,indent=4))


{
    "scenario": "e_commerce_33",
    "tables_count": 19,
    "inserts_count": 133,
    "sample_data": {
        "tables": [
            {
                "table_name": "users",
                "reasoning": "Creates the primary authenticated user (id=1) and another user to support multi-user scenarios like reviews and marketplace sellers. User 1 owns carts, orders, wishlists, addresses, payment methods, devices, subscriptions, and reviews required by the tasks. Timestamps are valid ISO 8601 expressions. All columns comply with the schema and uniqueness constraints on username and email.",
                "insert_statements": [
                    "INSERT INTO users (id, username, email, full_name, created_at, updated_at) VALUES (1, 'primary_user', 'primary_user@example.com', 'Primary User', datetime('now', '-365 days'), datetime('now', '-1 days'));",
                    "INSERT INTO users (id, username, email, full_name, created_at, updated_at) VALUES (2, 'secondary_user', 'secondary_

In [26]:
data_item_awm_sample.keys()

dict_keys(['scenario', 'tables_count', 'inserts_count', 'sample_data'])

In [27]:
data_path = "/share/home/sxjiang/dataset/AgentWorldModel-1K/gen_scenario.jsonl"
dataset_awm_scenario = datasets.load_dataset("json", data_files=data_path)
data_item_awm_scenario = dataset_awm_scenario['train'][0]
print(json.dumps(data_item_awm_scenario,indent=4))


{
    "name": "marketplace_1",
    "description": "VolunteerMatch is a platform connecting volunteers with nonprofit volunteer opportunities. Core entities include volunteer profiles, organizations, opportunities, applications, shifts, and skills. Nonprofits create organization profiles and post volunteer opportunities specifying roles, required skills, time commitment, location or virtual status, and schedules. They can define shifts with dates, times, and capacity per shift. Volunteers create profiles with interests, skills, availability, and preferred causes, and can search and apply to opportunities. Applications can be accepted, rejected, or waitlisted, and applicants receive status updates. Opportunity owners manage rosters per shift, track attendance, and log volunteer hours. Volunteers can update their availability, confirm shift attendance, and maintain a historical record of hours served, downloadable as a report. Messaging workflows support communication between organization

In [28]:
data_item_awm_scenario.keys()


dict_keys(['name', 'description'])

In [29]:
data_path = "/share/home/sxjiang/dataset/AgentWorldModel-1K/gen_spec.jsonl"
dataset_awm_spec = datasets.load_dataset("json", data_files=data_path)
data_item_awm_spec = dataset_awm_spec['train'][0]
print(json.dumps(data_item_awm_spec,indent=4))


{
    "scenario": "e_commerce_33",
    "api_spec": {
        "api_groups": [
            {
                "group_name": "Products",
                "endpoints": [
                    {
                        "path": "/api/products/search",
                        "method": "GET",
                        "summary": "Search products with text, filters, and sorting",
                        "description": "Search products by text and optional filters like category, Prime, rating, price, and sort order.",
                        "operation_id": "search_products",
                        "tags": [
                            "products",
                            "search"
                        ],
                        "request_params": {
                            "query": {
                                "type": "string",
                                "param_type": "query",
                                "required": false,
                                "description": "Full-te

In [30]:
data_path = "/share/home/sxjiang/dataset/AgentWorldModel-1K/gen_tasks.jsonl"
dataset_awm_tasks = datasets.load_dataset("json", data_files=data_path)
data_item_awm_tasks = dataset_awm_tasks['train'][0]
print(json.dumps(data_item_awm_tasks,indent=4))



{
    "scenario": "e_commerce_33",
    "tasks": [
        "Search for 'wireless noise cancelling headphones', sort results by average customer rating, and add the top-rated item under $200 to my cart in quantity 1.",
        "From my past orders, find the most recent purchase of 'paper towels' and reorder the exact same item and quantity.",
        "Create a new wish list named 'Kitchen Upgrades 2025' and add the three best-selling air fryers under $150 to that list.",
        "Update my default shipping address to '1234 Elm Street, Apt 5B, Springfield, IL 62704' and set it as the primary address for all future orders.",
        "Find a Kindle eBook version of 'Atomic Habits' by James Clear, ensure the price is under $20, and purchase it to be delivered to my default Kindle device.",
        "Search for 'USB-C charging cable 6ft', filter results to only show items with Prime eligible shipping and at least a 4-star rating, then add the cheapest option to my cart with quantity 2.",
     

In [31]:
data_path = "/share/home/sxjiang/dataset/AgentWorldModel-1K/gen_verifier.jsonl"
dataset_awm_verifier = datasets.load_dataset("json", data_files=data_path)
data_item_awm_verifier = dataset_awm_verifier['train'][0]
print(json.dumps(data_item_awm_verifier,indent=4))



{
    "scenario": "e_commerce_33",
    "task_idx": 0,
    "task": "Search for 'wireless noise cancelling headphones', sort results by average customer rating, and add the top-rated item under $200 to my cart in quantity 1.",
    "verification": {
        "code": "def verify_task(initial_db_path: str, final_db_path: str) -> dict:\n    import sqlite3\n\n    def dict_factory(cursor, row):\n        return {col[0]: row[idx] for idx, col in enumerate(cursor.description)}\n\n    def get_connection(db_path):\n        conn = sqlite3.connect(db_path)\n        conn.row_factory = dict_factory\n        return conn\n\n    def fetchone(conn, query, params=None):\n        cur = conn.cursor()\n        cur.execute(query, params or [])\n        return cur.fetchone()\n\n    def fetchall(conn, query, params=None):\n        cur = conn.cursor()\n        cur.execute(query, params or [])\n        return cur.fetchall()\n\n    # Connect to both database states\n    conn_initial = get_connection(initial_db_path)\

In [32]:
data_path = "/share/home/sxjiang/dataset/AgentWorldModel-1K/gen_verifier.pure_code.jsonl"
dataset_awm_verifier_pure_code = datasets.load_dataset("json", data_files=data_path)
data_item_awm_verifier_pure_code = dataset_awm_verifier_pure_code['train'][0]
print(json.dumps(data_item_awm_verifier_pure_code,indent=4))

data_item_awm_verifier_pure_code.keys()


{
    "scenario": "e_commerce_33",
    "task_idx": 0,
    "task": "Search for 'wireless noise cancelling headphones', sort results by average customer rating, and add the top-rated item under $200 to my cart in quantity 1.",
    "verification": {
        "code": "def verify_task_completion(initial_db_path: str, final_db_path: str, final_answer: str | None = None) -> dict:\n    import sqlite3\n    import re\n\n    def safe_query(db_path: str, query: str, params: tuple = ()) -> list:\n        \"\"\"Safely execute a read-only query on the given SQLite database.\n\n        Returns an empty list on any error.\n        \"\"\"\n        try:\n            conn = sqlite3.connect(f\"file:{db_path}?mode=ro\", uri=True)\n            cur = conn.cursor()\n            cur.execute(query, params)\n            rows = cur.fetchall()\n            conn.close()\n            return rows\n        except Exception:\n            return []\n\n    # 1. Find primary user's active cart (user_id = 1) in initial and f

dict_keys(['scenario', 'task_idx', 'task', 'verification'])

In [33]:
def ue1jfd():
    print("123")

In [34]:
ue1jfd()

123
